# Black Signal

В каждой спектрограмме закодирована последовательность из пяти цифр. Этот baseline читает PNG как одноканальное изображение, обучает небольшую CNN и формирует `submission.csv`.

При запуске ноутбук создаёт два файла:

- `fresh_checkpoint.pt` — веса после одной эпохи обучения в текущем запуске;
- `submission.csv` — предсказания для тестовой выборки.

Для инференса используется `datasets/solution/model.pt` — ваш заранее полностью обученный checkpoint, добавленный в workspace как отдельный набор данных.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset

SEED = 42
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
TARGET_WIDTH = 384

DATA_DIR = Path("datasets")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

FRESH_CHECKPOINT_PATH = Path("fresh_checkpoint.pt")
FULL_CHECKPOINT_PATH = DATA_DIR / "solution" / "model.pt"
SUBMISSION_PATH = Path("submission.csv")

random.seed(SEED)
np.random.seed(SEED)
_ = torch.manual_seed(SEED)

## Данные

Спектрограмма переводится в grayscale, разворачивается по частотной оси, нормализуется в диапазон `[0, 1]` и дополняется нулями или обрезается до ширины `384`. Колонка с именем PNG может называться `id` или `file`.

In [ ]:
def image_path(directory: Path, image_id: str) -> Path:
    path = directory / str(image_id)
    if path.suffix.lower() != ".png":
        path = path.with_suffix(".png")
    return path


def load_spectrogram(path: Path) -> torch.Tensor:
    image = np.asarray(Image.open(path).convert("L"), dtype=np.float32)
    image = image[::-1] / 255.0

    padding = max(0, TARGET_WIDTH - image.shape[1])
    image = np.pad(image, ((0, 0), (0, padding)))[:, :TARGET_WIDTH]
    image = np.ascontiguousarray(image)

    return torch.from_numpy(image).unsqueeze(0)


class SpectrogramDataset(Dataset):
    def __init__(self, table: pd.DataFrame, directory: Path):
        self.table = table.reset_index(drop=True)
        self.directory = directory
        self.image_column = "file" if "file" in table.columns else "id"

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, index: int):
        row = self.table.iloc[index]
        image = load_spectrogram(image_path(self.directory, row[self.image_column]))
        target = torch.tensor([int(digit) for digit in row["code"]], dtype=torch.long)
        return image, target


train_df = pd.read_csv(TRAIN_DIR / "labels.csv", dtype=str)
test_df = pd.read_csv(TEST_DIR / "labels.csv", dtype=str)
train_df["code"] = train_df["code"].str.zfill(5)

train_dataset = SpectrogramDataset(train_df, TRAIN_DIR)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_df)} samples")
print(f"Test:  {len(test_df)} samples")

## Модель и одна эпоха обучения

Модель выдаёт тензор размера `5 × 10`: для каждой из пяти позиций — логиты десяти цифр. Свежий checkpoint сохраняется сразу после одной эпохи.

In [ ]:
def build_model() -> nn.Module:
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(4),
        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d((4, 8)),
        nn.Flatten(),
        nn.Linear(32 * 4 * 8, 5 * 10),
        nn.Unflatten(1, (5, 10)),
    )


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> float:
    model.train()
    total_loss = 0.0

    for images, targets in loader:
        optimizer.zero_grad()

        logits = model(images)
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, 10),
            targets.reshape(-1),
        )

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(images)

    return total_loss / len(loader.dataset)


fresh_model = build_model()
optimizer = torch.optim.Adam(fresh_model.parameters(), lr=LEARNING_RATE)

train_loss = train_one_epoch(fresh_model, train_loader, optimizer)
torch.save(fresh_model.state_dict(), FRESH_CHECKPOINT_PATH)

print(f"Train loss: {train_loss:.4f}")
print(f"Saved: {FRESH_CHECKPOINT_PATH}")

## Инференс и submission

Для итогового предсказания загружаются полностью обученные веса из `datasets/solution/model.pt`. Функция `predict` принимает путь к одному PNG и возвращает строку из пяти цифр.

In [ ]:
model = build_model()
model.load_state_dict(torch.load(FULL_CHECKPOINT_PATH, map_location="cpu"))
model.eval()


@torch.inference_mode()
def predict(png_path: str | Path) -> str:
    image = load_spectrogram(Path(png_path)).unsqueeze(0)
    digits = model(image).argmax(dim=-1).squeeze(0).tolist()
    return "".join(map(str, digits))


test_image_column = "file" if "file" in test_df.columns else "id"
submission_ids = test_df["id"] if "id" in test_df.columns else test_df[test_image_column]

submission = pd.DataFrame(
    {
        "id": submission_ids,
        "code": [
            predict(image_path(TEST_DIR, image_id))
            for image_id in test_df[test_image_column]
        ],
    }
)
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"Saved: {SUBMISSION_PATH}")
submission.head()